# JSDP Poster Generator — A0 portrait

Fills `poster_template.html` with numbers read from the result files and writes a finished poster.

**To make Draft 2:** edit **Cell 1 (Settings)** only, then Run All.

**Output:** `poster/poster_<DRAFT_LABEL>.html` — A0 portrait, 841 × 1189 mm.

**PDF:** open the output in Chrome → Print → Paper **A0** → Margins **None** → Scale **100%** → **Background graphics ON** (required, or the colour panels print white).

Requires: `pandas`, `qrcode`, `Pillow`.

---
## Cell 1 — Settings

This is the only cell you edit.

In [ ]:
from pathlib import Path

REPO = Path("/Users/kaisar/workspace/geneformer-lung-tcell")
OUT  = REPO / "poster"

# ---- label for this run -------------------------------------------------
DRAFT_LABEL = "Draft 2"

# ---- title block --------------------------------------------------------
TITLE   = ("Foundation Model Analysis Reveals Distinct T-Cell Dysfunction Programs "
           "in Small Cell Lung Cancer Compared with Lung Adenocarcinoma and Normal Lung Tissue")
AUTHOR1 = "Kaisar Dauyey"
AUTHOR2 = "Shinji Nakaoka"
AFFIL   = ("Laboratory of Mathematical Biology, Faculty of Advanced Life Science, "
           "Hokkaido University, Japan")
EMAIL       = "snakaoka@sci.hokudai.ac.jp"
ABSTRACT_NO = "P25"
CONF_NAME   = "24JSDP"
COI         = "The authors declare no conflict of interest."
GITHUB_URL  = "https://github.com/Kays3/geneformer-lung-tcell"

# ---- result files (numbers are read from these) -------------------------
F_METRICS   = REPO / "sclc_validation/perturbation_workflow/results/test_metrics.json"
F_CONFUSION = REPO / "sclc_validation/perturbation_workflow/results/test_confusion_matrix.csv"
F_COHORT    = REPO / "sclc_validation/perturbation_workflow/results/htan_cohort_characterization_summary.csv"
F_HITS      = REPO / "sclc_validation/perturbation_workflow/targeted_panel/results/targeted_panel_concordant_hits_with_donor_robustness.csv"
F_SPATIAL   = REPO / "sclc_validation/spatial_validation/results/spatial_tcell_dysfunction_correlation_by_sample.csv"
F_POOLED    = REPO / "sclc_validation/spatial_validation/results/spatial_tcell_dysfunction_correlation_pooled.csv"

# ---- figures ------------------------------------------------------------
FIG_CONFUSION = REPO / "current_workflow/visuals/final_tcell_confusion_matrix.png"
FIG_SPATIAL   = REPO / "sclc_validation/spatial_validation/figures/spatial_tissue_validation_panel.png"
FIG_FOREST    = REPO / "sclc_validation/spatial_validation/figures/tcell_dysfunction_correlation_forest.png"

# ---- extra panels for new analyses --------------------------------------
# column: "left" | "middle" | "right"
# accent: "" | sky | green | amber | mid | hema | eosin
#         (hema = hematoxylin violet, eosin = rose - use for pathology content)
EXTRA_PANELS = [
    # {
    #     "column": "middle",
    #     "title": "Genome-scale perturbation screen",
    #     "accent": "sky",
    #     "text": "All-gene screen across held-out SCLC cells.",
    #     "figure": REPO / "path/to/figure.png",   # or None
    #     "caption": "Fig. 4 - Top genes by goal-state shift.",
    #     "table": REPO / "path/to/table.csv",     # or None
    #     "table_rows": 10,
    # },
]

# ---- prose blocks -------------------------------------------------------
# Numbers in braces are filled from the result files: {acc} {macro_f1}
# {n_concordant} {n_fully} {rho} {ci_low} {ci_high} {n_spots} {n_slides}
# {n_test_cells} {n_test_donors}
CONCLUSIONS = [
    "Geneformer-V2-104M fine-tuned on HTAN SCLC T cells reaches <strong>{acc} accuracy, "
    "macro F1 {macro_f1}</strong> on a donor-held-out test set.",

    "Bidirectional perturbation of the 21-gene panel yields <strong>{n_concordant} concordant "
    "hits</strong>, of which <strong>{n_fully} are fully donor-consistent</strong>, including "
    "TIGIT, HAVCR2, CTLA4 and IL7R.",

    "ASCL1 and NEUROD1 reproduce expected SCLC master regulator behaviour, confirming the "
    "perturbation pipeline recovers known biology.",

    "Spatial pathology on {n_slides} independent SCLC tumour sections ({n_spots} spots) gives "
    "pooled rho <strong>{rho} [{ci_low}, {ci_high}]</strong> between T-cell abundance and "
    "dysfunction marker expression, recovering the association in intact tissue.",

    "Foundation-model perturbation combined with quantitative spatial pathology gives a "
    "multi-layer framework for nominating immunotherapy targets in SCLC.",
]

LIMITATIONS = [
    "Thin normal class: 4 donors in total, 1 in the test split.",
    "No Normal or LUAD sections in the spatial cohort, so SCLC-specificity is untested.",
    "Spatial scores are marker-panel proxies, not deconvolved cell-type proportions.",
    "Perturbation shifts are model-derived, not experimental knockouts.",
    "Targeted 21-gene panel; the genome-scale SCLC screen is still pending.",
]

FUTURE = [
    ("Immediate", [
        "CellBender ambient RNA correction, then re-test the panel",
        "CD4 / CD8 subtype-stratified concordance",
        "Ribosomal gene sensitivity analysis",
        "GSEA on the fully donor-consistent hits",
    ]),
    ("Spatial pathology", [
        "cell2location deconvolution in place of marker scores",
        "Tumour versus stroma region segmentation",
        "Normal and LUAD sections as specificity controls",
        "Link dysfunction score to treatment response (GSE261348)",
    ]),
    ("Genome scale", [
        "All-gene in silico screen on held-out SCLC cells",
        "STRING network hub prioritisation",
        "Cross-disease comparison against melanoma exhaustion signatures",
        "scGPT cross-model replication",
    ]),
    ("Experimental", [
        "CRISPR knockout of TIGIT and HAVCR2 in SCLC co-culture",
        "Perturb-seq in primary T cells versus in silico predictions",
        "Pseudotime and RNA velocity trajectory analysis",
        "Clinical outcome correlation once metadata is available",
    ]),
]

print("Settings loaded:", DRAFT_LABEL)

---
## Cell 2 — Read the result files

In [ ]:
import json
import pandas as pd

metrics   = json.loads(F_METRICS.read_text())
confusion = pd.read_csv(F_CONFUSION, index_col=0)
cohort    = pd.read_csv(F_COHORT)
hits      = pd.read_csv(F_HITS)
spatial   = pd.read_csv(F_SPATIAL)
pooled    = pd.read_csv(F_POOLED)

acc      = metrics["test_metrics"]["acc"]
macro_f1 = metrics["test_metrics"]["macro_f1"]

test_rows     = cohort[cohort["split"] == "test"]
n_test_cells  = int(test_rows["n_cells"].sum())
n_test_donors = int(test_rows["n_donors"].sum())

n_concordant = len(hits)
n_fully      = int((hits["donor_robustness"] == "fully_consistent").sum())

rho      = float(pooled["rho_pooled"].iloc[0])
ci_low   = float(pooled["ci_low"].iloc[0])
ci_high  = float(pooled["ci_high"].iloc[0])
n_spots  = int(spatial["n_spots"].sum())
n_sig    = int((spatial["p_value"] < 0.001).sum())
n_slides = len(spatial)

print(f"accuracy        {acc:.4f}")
print(f"macro F1        {macro_f1:.4f}")
print(f"test cells      {n_test_cells:,} from {n_test_donors} donors")
print(f"concordant      {n_concordant}  ({n_fully} fully donor-consistent)")
print(f"pooled rho      {rho:.3f} [{ci_low:.3f}, {ci_high:.3f}]")
print(f"spatial         {n_slides} specimens, {n_spots:,} spots, {n_sig} at p<0.001")

---
## Cell 3 — Helpers

Images are inlined as base64 so the finished HTML is one self-contained file.
A missing figure renders as a dashed "pending" box rather than breaking the build.

In [ ]:
import base64, io
import qrcode
from PIL import Image as PILImage

def img_tag(path, css_class="fig", alt="", width_mm=None):
    """Return an <img> with the file inlined, or a pending box if absent.

    width_mm: the rendered width of this image in millimetres, taken from the
    A0 grid geometry in poster_template.html. When given, an explicit
    height is computed from the image's real pixel aspect ratio and written
    as an inline style.

    This is required, not cosmetic: WeasyPrint 69 mis-paginates a multi-page
    PDF (observed: 1 page of content -> 5 mostly-blank pages) when an <img>
    inside this template's nested grid/flex layout has width set with no
    matching height. Browsers render the same markup with one page. Chrome's
    own headless PDF export is unavailable in this environment, so the
    notebook works around the WeasyPrint-specific bug at the markup level
    instead of depending on layout-engine behaviour that cannot be tested
    here. If you switch renderers, this workaround is unnecessary but
    harmless - object-fit:contain still displays correctly with a height.
    """
    if path is None or not Path(path).exists():
        name = Path(path).name if path else "figure"
        return f'<div class="pending">Pending: {name}</div>'
    raw = Path(path).read_bytes()
    b64 = base64.b64encode(raw).decode()
    cls = f' class="{css_class}"' if css_class else ""
    style = ""
    if width_mm is not None:
        w_px, h_px = PILImage.open(io.BytesIO(raw)).size
        height_mm = width_mm * h_px / w_px
        style = f' style="height:{height_mm:.1f}mm;object-fit:contain;"'
    return f'<img src="data:image/png;base64,{b64}"{cls}{style} alt="{alt}">'

def qr_tag(url):
    qr = qrcode.QRCode(version=2, box_size=8, border=2,
                       error_correction=qrcode.constants.ERROR_CORRECT_M)
    qr.add_data(url)
    qr.make(fit=True)
    buf = io.BytesIO()
    qr.make_image(fill_color="#10243c", back_color="white").save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    # QR sizing is fixed by .qr-wrap img{width:34mm;height:34mm} in the
    # template CSS, so no inline height is injected here.
    return f'<img src="data:image/png;base64,{b64}" alt="QR code">'

def li(items):
    return "\n".join(f"<li>{x}</li>" for x in items)

def df_to_rows(df, n=10):
    rows = []
    for _, r in df.head(n).iterrows():
        rows.append("<tr>" + "".join(f"<td>{v}</td>" for v in r) + "</tr>")
    return "\n".join(rows)

def df_to_header(df):
    return "".join(f"<th>{c}</th>" for c in df.columns)

# --- A0 grid geometry, mirrors poster_template.html --------------------
# Recomputed here (not imported) because the template is plain HTML/CSS, not
# a module. If the grid geometry changes in poster_template.html (column
# widths, panel padding, gaps), update these to match or images will be
# sized for the wrong column width.
_A0_W = 841
_PAD, _GAP = 7, 7
_LEFT_COL, _RIGHT_COL = 248, 248
_MID_COL = _A0_W - 2*_PAD - 2*_GAP - _LEFT_COL - _RIGHT_COL
_PANEL_PAD = 5
_MOUNT_PAD = 2.4
_MID_INNER = _MID_COL - 2*_PANEL_PAD

WIDTH_MM = {
    "cm":      (_MID_INNER - 3.5) / 2,        # .twocol, 2 equal columns
    "spatial": _MID_INNER - 2*_MOUNT_PAD,     # inside .slide-mount
    "forest":  (_MID_INNER - 3.5) * 3 / 5,    # .twocol-32, 3fr of 5
}

print("helpers ready")
print(f"  image widths (mm): {WIDTH_MM}")

---
## Cell 4 — Build the HTML fragments

In [ ]:
DISEASE_LABEL = {
    "small cell lung carcinoma": "SCLC",
    "lung adenocarcinoma": "LUAD",
    "normal": "Normal",
}
DISEASE_ORDER = ["small cell lung carcinoma", "lung adenocarcinoma", "normal"]

# ---- cohort table -------------------------------------------------------
cells_by  = cohort.pivot_table(index="disease", columns="split",
                               values="n_cells", aggfunc="sum")
donors_by = cohort.pivot_table(index="disease", columns="split",
                               values="n_donors", aggfunc="sum")
cohort_rows = []
for disease in DISEASE_ORDER:
    if disease not in cells_by.index:
        continue
    label = DISEASE_LABEL[disease]
    tr = int(cells_by.loc[disease].get("train", 0))
    ev = int(cells_by.loc[disease].get("eval", 0))
    te = int(cells_by.loc[disease].get("test", 0))
    dn = int(donors_by.loc[disease].sum())
    strong = ' class="bold"' if label == "SCLC" else ""
    cohort_rows.append(
        f"<tr><td{strong}>{label}</td><td class='num'>{tr:,}</td>"
        f"<td class='num'>{ev:,}</td><td class='num'>{te:,}</td>"
        f"<td class='num bold'>{dn}</td></tr>")
COHORT_ROWS = "\n".join(cohort_rows)

# ---- confusion matrix ---------------------------------------------------
cm_rows = []
for label in confusion.index:
    row = confusion.loc[label]
    total = row.sum()
    recall = f"{row[label] / total * 100:.1f}%" if total else "n/a"
    body = ""
    for col in confusion.columns:
        hl = " hl" if col == label else ""
        body += f"<td class='num{hl}'>{int(row[col]):,}</td>"
    cm_rows.append(f"<tr><td class='bold'>{label}</td>{body}"
                   f"<td class='num'>{recall}</td></tr>")
CM_ROWS = "\n".join(cm_rows)

# ---- concordant hits, SCLC-source, strongest deletion effect first ------
sclc_hits = (hits[hits["source_state"] == "sclc"]
             .sort_values("abs_delete_shift", ascending=False)
             .head(10))
hit_rows = []
for _, r in sclc_hits.iterrows():
    full  = r["donor_robustness"] == "fully_consistent"
    tag   = "full" if full else "partial"
    style = "color:var(--teal);font-weight:700" if full else "color:var(--amber)"
    hit_rows.append(
        f"<tr><td class='bold'>{r['Gene_name']}</td><td>{r['comparison']}</td>"
        f"<td class='num'>{r['delete_shift']:+.3f}</td>"
        f"<td class='num'>{r['overexpress_shift']:+.3f}</td>"
        f"<td class='num'>{int(r['delete_n']):,}</td>"
        f"<td style='{style}'>{tag}</td></tr>")
HIT_ROWS = "\n".join(hit_rows)

# ---- spatial per-specimen table -----------------------------------------
spatial_sorted = spatial.sort_values("rho", ascending=False)
sp_rows = []
for _, r in spatial_sorted.iterrows():
    sp_rows.append(
        f"<tr><td>{r['sample_label']}</td><td class='num'>{int(r['n_spots']):,}</td>"
        f"<td class='num'>{r['rho']:.3f}</td>"
        f"<td class='num'>[{r['ci_low']:.3f}, {r['ci_high']:.3f}]</td>"
        f"<td class='num'>{r['p_value']:.1e}</td></tr>")
sp_rows.append(
    f"<tr style='background:#f7eef1;font-weight:700'><td>Pooled</td>"
    f"<td class='num'>{n_spots:,}</td>"
    f"<td class='num' style='color:var(--eosin)'>{rho:.3f}</td>"
    f"<td class='num'>[{ci_low:.3f}, {ci_high:.3f}]</td>"
    f"<td class='num'>&lt;1e-99</td></tr>")
SPATIAL_ROWS = "\n".join(sp_rows)

# ---- slide tray: one card per specimen ----------------------------------
# Bar length encodes rho relative to the strongest specimen; colour marks
# whether that specimen individually reached p < 0.001.
rho_max = max(spatial_sorted["rho"].max(), 1e-9)
slide_cards = []
for _, r in spatial_sorted.iterrows():
    sig  = r["p_value"] < 0.001
    col  = "var(--eosin)" if sig else "#b9c4d2"
    txt  = "var(--eosin)" if sig else "var(--ink-2)"
    frac = max(r["rho"] / rho_max, 0.0)
    slide_cards.append(
        f"<div class='slide-card'>"
        f"<div class='sc-top'>{r['sample_label']}</div>"
        f"<div class='sc-rho' style='color:{txt}'>{r['rho']:.3f}</div>"
        f"<div class='sc-n'>{int(r['n_spots']):,} spots</div>"
        f"<div class='sc-bar' style='background:linear-gradient(90deg,"
        f"{col} {frac*100:.0f}%, #e6ebf1 {frac*100:.0f}%)'></div>"
        f"</div>")
SLIDE_CARDS = "\n".join(slide_cards)

# ---- prose, with numbers substituted ------------------------------------
numbers = dict(
    acc=f"{acc:.1%}", macro_f1=f"{macro_f1:.3f}",
    n_concordant=n_concordant, n_fully=n_fully,
    rho=f"{rho:.3f}", ci_low=f"{ci_low:.3f}", ci_high=f"{ci_high:.3f}",
    n_spots=f"{n_spots:,}", n_slides=n_slides,
    n_test_cells=f"{n_test_cells:,}", n_test_donors=n_test_donors,
)
CONCLUSIONS_HTML = li(c.format(**numbers) for c in CONCLUSIONS)
LIMITATIONS_HTML = li(LIMITATIONS)
FUTURE_HTML = "\n".join(
    f"<div class='fb'><h3>{heading}</h3><ul>{li(items)}</ul></div>"
    for heading, items in FUTURE)

# ---- extra panels -------------------------------------------------------
def render_panel(spec):
    parts = [f'<div class="panel {spec.get("accent", "")}">',
             f'<h2>{spec["title"]}</h2>']
    if spec.get("text"):
        parts.append(f'<p>{spec["text"]}</p>')
    if spec.get("figure"):
        parts.append(img_tag(spec["figure"]))
        if spec.get("caption"):
            parts.append(f'<div class="fcap">{spec["caption"]}</div>')
    if spec.get("table"):
        tpath = Path(spec["table"])
        if tpath.exists():
            tdf = pd.read_csv(tpath)
            parts.append(f'<table class="mt s"><tr>{df_to_header(tdf)}</tr>'
                         f'{df_to_rows(tdf, spec.get("table_rows", 10))}</table>')
        else:
            parts.append(f'<div class="pending">Pending: {tpath.name}</div>')
    parts.append("</div>")
    return "\n".join(parts)

panels_by_column = {"left": [], "middle": [], "right": []}
for spec in EXTRA_PANELS:
    panels_by_column[spec.get("column", "middle")].append(render_panel(spec))

print(f"cohort rows   {len(cohort_rows)}")
print(f"hit rows      {len(hit_rows)}")
print(f"spatial rows  {len(sp_rows)}")
print(f"slide cards   {len(slide_cards)}")
print(f"extra panels  left={len(panels_by_column['left'])} "
      f"middle={len(panels_by_column['middle'])} right={len(panels_by_column['right'])}")

---
## Cell 5 — Fill the template and write the file

In [ ]:
import re

template = (OUT / "poster_template.html").read_text(encoding="utf-8")

replacements = {
    "TITLE": TITLE, "AUTHOR1": AUTHOR1, "AUTHOR2": AUTHOR2, "AFFIL": AFFIL,
    "EMAIL": EMAIL, "ABSTRACT_NO": ABSTRACT_NO, "CONF_NAME": CONF_NAME,
    "COI": COI, "GITHUB_URL": GITHUB_URL, "DRAFT_LABEL": DRAFT_LABEL,

    "ACC": f"{acc:.1%}", "MACRO_F1": f"{macro_f1:.3f}",
    "N_TEST_CELLS": f"{n_test_cells:,}", "N_TEST_DONORS": str(n_test_donors),
    "N_CONCORDANT": str(n_concordant), "N_FULLY": str(n_fully),
    "RHO": f"{rho:.3f}", "CI_LOW": f"{ci_low:.3f}", "CI_HIGH": f"{ci_high:.3f}",
    "N_SPOTS": f"{n_spots:,}", "N_SIG_SAMPLES": f"{n_sig}/{n_slides}",
    "N_SLIDES": str(n_slides),

    "COHORT_ROWS": COHORT_ROWS, "CM_ROWS": CM_ROWS,
    "HIT_ROWS": HIT_ROWS, "SPATIAL_ROWS": SPATIAL_ROWS,
    "SLIDE_CARDS": SLIDE_CARDS,
    "CONCLUSIONS": CONCLUSIONS_HTML, "LIMITATIONS": LIMITATIONS_HTML,
    "FUTURE": FUTURE_HTML,

    "IMG_CM":      img_tag(FIG_CONFUSION, alt="Confusion matrix", width_mm=WIDTH_MM["cm"]),
    "IMG_SPATIAL": img_tag(FIG_SPATIAL, css_class="", alt="Visium spatial score maps",
                          width_mm=WIDTH_MM["spatial"]),
    "IMG_FOREST":  img_tag(FIG_FOREST, alt="Forest plot", width_mm=WIDTH_MM["forest"]),
    "IMG_QR":      qr_tag(GITHUB_URL),

    "EXTRA_LEFT_PANELS":   "\n".join(panels_by_column["left"]),
    "EXTRA_MIDDLE_PANELS": "\n".join(panels_by_column["middle"]),
    "EXTRA_RIGHT_PANELS":  "\n".join(panels_by_column["right"]),
}

html = template
for key, value in replacements.items():
    html = html.replace("{{" + key + "}}", str(value))

# fail loudly if the template has a token the notebook does not fill
unfilled = sorted(set(re.findall(r"\{\{([A-Z_]+)\}\}", html)))
assert not unfilled, f"unfilled tokens in template: {unfilled}"

slug = DRAFT_LABEL.lower().replace(" ", "_")
out_path = OUT / f"poster_{slug}.html"
out_path.write_text(html, encoding="utf-8")

print(f"written  {out_path}")
print(f"size     {out_path.stat().st_size / 1024:.0f} KB")
print()
print("PDF export:")
print("  Chrome > Print > Paper A0 > Margins None > Scale 100%")
print("  Background graphics MUST be ON or the colour panels print white.")

---
## Cell 6 — Preview inline (optional)

---
## Cell 6 — Export a print-ready A0 PDF

Uses WeasyPrint, which reads the template's `@page { size: A0 portrait }`
rule directly, so the output PDF is exactly 841 x 1189 mm with no browser
print-dialog steps.

Requires a WeasyPrint environment with its system libraries (Pango, Cairo,
GDK-Pixbuf) installed — a plain `pip install weasyprint` is not enough on
most systems. If `import weasyprint` fails, install those first (e.g. via
conda-forge: `conda install -c conda-forge weasyprint pango cairo gdk-pixbuf`),
or fall back to the browser method below.

**Why this cell exists:** every `<img>` in this template gets an inline
`height` from `img_tag(..., width_mm=...)` in Cell 3. Without it, WeasyPrint
mis-paginates the poster into several mostly-blank pages instead of one A0
sheet — a WeasyPrint-specific layout bug with `width:100%`-only images
inside nested grid/flex containers, not a sign the content overflows.
`--check-pages` below fails loudly if that ever regresses.

In [ ]:
import subprocess

pdf_out = out_path.with_suffix(".pdf")

try:
    import weasyprint
    weasyprint.HTML(filename=str(out_path)).write_pdf(str(pdf_out))
    print(f"written  {pdf_out}  ({pdf_out.stat().st_size / 1e6:.1f} MB)")
except ImportError:
    print("WeasyPrint not available in this environment.")
    print("Fallback: open the HTML in Chrome, Print > Paper A0 > Margins None")
    print("> Scale 100% > Background graphics ON > Save as PDF.")
    pdf_out = None

# --check-pages: confirm the export landed on exactly one A0 page.
# A page count > 1 here means the WeasyPrint pagination bug described above
# has resurfaced - check that every <img> in the generated HTML still carries
# an explicit inline height (grep for `alt=` without `height:` nearby).
if pdf_out and pdf_out.exists():
    import pypdfium2 as pdfium
    doc = pdfium.PdfDocument(pdf_out)
    n_pages = len(doc)
    page = doc[0]
    w_mm, h_mm = page.get_width() * 25.4 / 72, page.get_height() * 25.4 / 72
    doc.close()
    print(f"pages: {n_pages}   page size: {w_mm:.0f} x {h_mm:.0f} mm")
    assert n_pages == 1, (
        f"expected 1 A0 page, got {n_pages} - an <img> is likely missing its "
        f"inline height again; see the WIDTH_MM injection in Cell 3")
    assert abs(w_mm - 841) < 2 and abs(h_mm - 1189) < 2, "page size drifted from A0"
    print("OK: single A0 page, correct dimensions.")

In [ ]:
from IPython.display import IFrame
IFrame(src=out_path.as_uri(), width="100%", height=1000)